# Part 1 : Practise

## Task 1 : Load and look

In [1]:
import pandas as pd

loans_raw = pd.read_csv("data/loans.csv")  # original : Never touch it
loans = loans_raw.copy()  # working copy to clean

In [2]:
loans.shape

(623, 9)

In [3]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


Problems found:

- loan_date and return_date are text, they should be dates
- late_fee is text, it should be a number
- returned is text, it should be True/False
- notes is completely empty (0 non-null), so it should be dropped
- return_date has 30 missing because those books are not returned yet

## Task 2 : Clean the genre

In [4]:
loans["genre"].nunique()

28

In [5]:
loans["genre"].value_counts()

genre
Crime              38
Crime              35
crime              33
Romance            32
Children           31
Children's         31
fantasy            29
children           29
 crime             29
CRIME              28
children's         28
history            22
Fantasy            22
Romance            20
 Fantasy           20
romance            19
FANTASY            19
science fiction    19
sci fi             17
Children           16
Sci-fi             16
 history           16
 ROMANCE           15
sci-fi             14
History            14
Science fiction    14
Sci fi             10
History             7
Name: count, dtype: int64

In [6]:
type(loans)

pandas.DataFrame

In [7]:
loans["genre"].nunique()

28

In [8]:
loans["genre"].unique()

<StringArray>
[       ' ROMANCE',           'Crime',         'fantasy',           'CRIME',
       'Children ',        'children',          'Crime ',      'children's',
         'history',         'romance',          'sci-fi',        'Children',
         'Fantasy',          'Sci-fi',        'History ',         'Romance',
 'Science fiction',          'Sci fi',           'crime',        'Romance ',
         'FANTASY',          ' crime',        ' Fantasy',         'History',
        ' history',      'Children's',         'sci fi ', 'science fiction']
Length: 28, dtype: str

In [9]:
loans["genre"] = loans["genre"].str.strip().str.lower()

In [10]:
loans["genre"].nunique()

9

In [11]:
loans["genre"].unique()

<StringArray>
[        'romance',           'crime',         'fantasy',        'children',
      'children's',         'history',          'sci-fi', 'science fiction',
          'sci fi']
Length: 9, dtype: str

In [12]:
# replace same name repitition

loans["genre"] = loans["genre"].replace({
    "children's" : "children",
    "science fiction" : "sci-fi",
    "sci fi" : "sci-fi"
})

In [13]:
loans["genre"].unique()

<StringArray>
['romance', 'crime', 'fantasy', 'children', 'history', 'sci-fi']
Length: 6, dtype: str

In [14]:
loans["genre"] = loans["genre"].str.title()

In [15]:
loans["genre"].unique()

<StringArray>
['Romance', 'Crime', 'Fantasy', 'Children', 'History', 'Sci-Fi']
Length: 6, dtype: str

Notes:

### `.replace` vs `.str.replace`

| | What it replaces | Works on |
|---|---|---|
| `.replace({...})` | the **whole cell** (the value must match exactly) | any column: text, numbers, anything |
| `.str.replace("a", "b")` | **part of the text inside** each cell | text only |

```python
s = pd.Series(["söder", "södermalm", "östermalm"])

s.replace({"söder": "södermalm"})
# 'södermalm', 'södermalm', 'östermalm'
#  only the cell that is exactly "söder" changed

s.str.replace("söder", "SÖDER")
# 'SÖDER', 'SÖDERmalm', 'östermalm'
#  every place "söder" appears INSIDE a cell changed
```

- `.replace` asks of every cell: is it **exactly** this value? If yes, swap it.
- `.str.replace` acts like Find & Replace: it changes the text **wherever** it appears inside a cell.

In [16]:
loans["genre"].value_counts()

genre
Crime       163
Children    135
Fantasy      90
Sci-Fi       90
Romance      86
History      59
Name: count, dtype: int64

### Answer – Task 2

**Problem:** `genre` had **28** different values, but the library only has 6 genres. The same genre was written in many ways: extra spaces (`' crime'`, `'Children '`), different capitals (`Crime`, `crime`, `CRIME`) and different words for the same thing (`sci fi`, `science fiction`).

**What I did:**
1. `.str.strip().str.lower()` removed the spaces and made everything small letters. **28 → 9** values.
2. `.replace({...})` put each synonym under one name: `children's` → `children`, `sci fi` and `science fiction` → `sci-fi`. **9 → 6** values.
3. `.str.title()` made the names look nice for reports (`sci-fi` → `Sci-Fi`).

Each step was saved back into `loans["genre"]`.

**Result:** 6 genres

| Genre | Loans |
|---|---|
| Crime | 163 |
| Children | 135 |
| Fantasy | 90 |
| Sci-Fi | 90 |
| Romance | 86 |
| History | 59 |

**Why it matters:** before cleaning, counting or grouping by genre would have split one genre into many small pieces. For example, Crime would have shown up as 5 separate rows.

## Task 3 : Yes and no

In [17]:
loans["returned"].value_counts(dropna=False)

returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64

In [18]:
loans["returned"] = loans["returned"].str.strip().str.lower()
loans["returned"].value_counts()

returned
yes    349
y      244
no      16
n       14
Name: count, dtype: int64

In [19]:
loans["returned"] = loans["returned"].map({"yes": True, "y": True, "no": False, "n": False})
loans["returned"].value_counts()

returned
True     593
False     30
Name: count, dtype: int64

### Answer – Task 3

**Problem:** `returned` was text with **10** different spellings: `Y`, `Yes`, `YES`, `y`, `yes` for yes and `N`, `NO`, `No`, `n`, `no` for no. It should be True/False. No values were missing.

**What I did:**
1. `.str.strip().str.lower()` made the 10 spellings into 4: `y`, `yes`, `n`, `no`.
2. `.map({"yes": True, "y": True, "no": False, "n": False})` turned those 4 into real True/False values.

**Result:**

| returned | Loans |
|---|---|
| True | 593 |
| False | 30 |

There is no NaN, so every value was mapped. 593 + 30 = 623, which matches all the rows.

**Why it matters:** as True/False, I can count returned books with `.sum()` and get the share with `.mean()`.

## Task 4 : The late fee is text

In [20]:
loans["late_fee"].unique()

<StringArray>
[  '7,50 kr',   '0,00 kr',  ' 0,00 kr',  '15,00 kr',   '35,00kr', ' 27,50 kr',
   '20,00kr',  '42,50 kr',   '2,50 kr',  '20,00 kr',  '45,00 kr',    '0,00kr',
 ' 37,50 kr', ' 22,50 kr', ' 35,00 kr',  '37,50 kr', ' 10,00 kr', ' 45,00 kr',
  '40,00 kr',   '47,50kr',   '27,50kr',  '17,50 kr',  '10,00 kr',  '27,50 kr',
 ' 30,00 kr',  ' 5,00 kr',   '15,00kr',  '30,00 kr',  '25,00 kr',  '47,50 kr',
  '35,00 kr',   '5,00 kr',   '17,50kr',   '25,00kr', ' 20,00 kr',    '2,50kr',
   '12,50kr', ' 42,50 kr',    '5,00kr', ' 47,50 kr',   '22,50kr',  '32,50 kr',
  '12,50 kr', ' 15,00 kr',  ' 2,50 kr',   '32,50kr',   '42,50kr',    '7,50kr',
  '22,50 kr', ' 12,50 kr', ' 17,50 kr',  ' 7,50 kr', ' 25,00 kr',   '40,00kr',
   '37,50kr',   '45,00kr',   '30,00kr',   '10,00kr', ' 40,00 kr']
Length: 59, dtype: str

In [21]:
# loans["late_fee"].astype(float)
# got ValueError: could not convert string to float: '7,50 kr'

In [22]:
loans["late_fee"]= loans["late_fee"].str.replace("kr", "")
loans["late_fee"]

0        7,50 
1        0,00 
2        0,00 
3        0,00 
4        0,00 
        ...   
618      0,00 
619      25,00
620     40,00 
621      0,00 
622      0,00 
Name: late_fee, Length: 623, dtype: str

In [23]:
loans["late_fee"] = loans["late_fee"].str.strip()
loans["late_fee"]

0       7,50
1       0,00
2       0,00
3       0,00
4       0,00
       ...  
618     0,00
619    25,00
620    40,00
621     0,00
622     0,00
Name: late_fee, Length: 623, dtype: str

In [24]:
loans["late_fee"]= loans["late_fee"].str.replace(",", ".")
loans["late_fee"]

0       7.50
1       0.00
2       0.00
3       0.00
4       0.00
       ...  
618     0.00
619    25.00
620    40.00
621     0.00
622     0.00
Name: late_fee, Length: 623, dtype: str

In [25]:
loans["late_fee"].astype(float)

0       7.5
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
618     0.0
619    25.0
620    40.0
621     0.0
622     0.0
Name: late_fee, Length: 623, dtype: float64

In [26]:
loans["late_fee"].describe()

count      623
unique      20
top       0.00
freq       321
Name: late_fee, dtype: object

In [27]:
loans["late_fee"].unique()

<StringArray>
[ '7.50',  '0.00', '15.00', '35.00', '27.50', '20.00', '42.50',  '2.50',
 '45.00', '37.50', '22.50', '10.00', '40.00', '47.50', '17.50', '30.00',
  '5.00', '25.00', '12.50', '32.50']
Length: 20, dtype: str

In [28]:
loans["late_fee"] = loans["late_fee"].astype(float)

In [29]:
loans["late_fee"].unique()

array([ 7.5,  0. , 15. , 35. , 27.5, 20. , 42.5,  2.5, 45. , 37.5, 22.5,
       10. , 40. , 47.5, 17.5, 30. ,  5. , 25. , 12.5, 32.5])

In [30]:
loans["late_fee"].describe()

count    623.000000
mean      12.323435
std       16.106338
min        0.000000
25%        0.000000
50%        0.000000
75%       25.000000
max       47.500000
Name: late_fee, dtype: float64

### Answer – Task 4

**Problem:** `late_fee` was text like `'7,50 kr'`, so I couldn't sum or average it. `astype(float)` gave
`ValueError: could not convert string to float: '7,50 kr'`. Two problems: the letters `kr` and a Swedish decimal comma. Some cells also had no space (`'25,00kr'`).

**What I did:**
1. `.str.replace(...)` removed `kr` and changed the comma `,` to a dot `.`.
2. `unique()` showed 20 clean values like `'7.50'`, still text.
3. `.astype(float)` turned them into real decimal numbers (`float64`).

**Result:**

| | Value |
|---|---|
| min | 0 |
| max | 47.5 |
| mean | 12.32 |
| total | 7677.5 |

**Note:** the median is 0, so more than half of the loans had no late fee. A few big fees (up to 47.5 kr) pull the mean up to 12.32.

## Task 5 : Dates

In [31]:
loans["loan_date"].sample(15)

533    2026-06-04
317    2026-04-03
528    2026-06-02
391    2026-04-24
45     2026-01-18
208    2026-03-04
172    2026-02-22
619    2026-06-26
364    2026-04-16
1      2026-01-05
197    2026-03-02
276    2026-03-24
230    2026-03-11
239    2026-03-12
607    2026-06-24
Name: loan_date, dtype: str

In [32]:
loans["loan_date"].count()

np.int64(623)

In [33]:
# pd.to_datetime() turns text into real dates
# errors="coerce"	if a value can’t be read as a date, put NaT instead of stopping with an error
# NaT	“Not a Time”. It’s NaN for dates: an empty date cell

loans["loan_date"] = pd.to_datetime(loans["loan_date"], errors='coerce')

In [34]:
loans["loan_date"].isna().sum()

np.int64(3)

In [35]:
loans["loan_date"].describe()

count                           620
mean     2026-04-01 18:23:13.548387
min             2026-01-05 00:00:00
25%             2026-02-17 00:00:00
50%             2026-04-01 12:00:00
75%             2026-05-15 06:00:00
max             2026-06-27 00:00:00
Name: loan_date, dtype: object

In [36]:
loans["loan_date"].min()

Timestamp('2026-01-05 00:00:00')

In [37]:
loans["loan_date"].max()

Timestamp('2026-06-27 00:00:00')

In [38]:
loans["loan_date"].dt.to_period("M").value_counts().sort_index()

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64

### Answer – Task 5

**Problem:** `loan_date` was text (`'2026-03-30'`), so pandas couldn't sort by date or group by month.

**What I did:**
1. `pd.to_datetime(..., errors="coerce")` turned the text into real dates (`datetime64`). Values that couldn't be read as a date became **NaT** (Not a Time, the empty date) instead of stopping with an error.
2. `isna().sum()` counted them: **3** dates became NaT.
3. `describe()` gave the first and last date.
4. `.dt.to_period("M")` gave each loan its month, then `value_counts().sort_index()` counted loans per month in time order.

**Result:**
- First loan: **2026-01-05**, last loan: **2026-06-27**

| Month | Loans |
|---|---|
| 2026-01 | 96 |
| 2026-02 | 100 |
| 2026-03 | 111 |
| 2026-04 | 106 |
| 2026-05 | 108 |
| 2026-06 | 99 |

Total 620 = 623 − 3 NaT. The NaT rows have no month, so they are not counted.

## Task 6 : Duplicates

In [39]:
# raw table
loans_raw.duplicated().sum()

np.int64(12)

In [40]:
# after cleaning
loans.duplicated().sum()

np.int64(18)

In [41]:
# raw table
loans_raw.duplicated(subset="loan_id").sum()

np.int64(18)

In [42]:
# after cleaning
loans.duplicated(subset="loan_id").sum()

np.int64(18)

In [43]:
fee_before = loans["late_fee"].sum()
fee_before

np.float64(7677.5)

In [44]:
loans = loans.drop_duplicates()

In [45]:
loans = loans.reset_index(drop=True)

In [46]:
loans.shape

(605, 9)

In [47]:
loans["late_fee"].sum()

np.float64(7447.5)

In [48]:
fee_diff = fee_before - loans["late_fee"].sum()

In [49]:
fee_diff / fee_before * 100

np.float64(2.9957668511885376)

### Answer – Task 6

**Problem:** some loans were saved more than once. Duplicates make every count and total too big.

**What I did:**
1. `duplicated().sum()` counted full-row duplicates (all columns compared):
   - raw table: **12**
   - after cleaning: **18**
2. `duplicated(subset="loan_id").sum()` checked only the loan ID: **18**.
3. `drop_duplicates()` removed the copies, then `reset_index(drop=True)` renumbered the rows 0, 1, 2 ...
4. `shape` after removing: **(605, 9)** = 623 − 18.
5. I saved the late fee total in `fee_before` **before** removing, then compared it with the total after.

**Why is the number different?** Cleaning made hidden copies visible. Before cleaning, two copies of the same loan could look different, for example `"Crime "` vs `"crime"`. After strip/lower/title they became identical, so 6 more duplicates showed up.

**subset="loan_id":** gives the same 18 as the full-row check on the cleaned table. So the duplicates are identical in every column (true copies), and a plain `drop_duplicates()` is safe.

**Effect on late fees:**

| | Late fee total |
|---|---|
| Before removing | 7677.5 kr |
| After removing | 7447.5 kr |
| Added by duplicates | **230 kr ≈ 3.0 %** |

## Task 7 : Put it together

In [50]:
loans = loans.drop(columns="notes")
loans.columns

Index(['loan_id', 'member_id', 'title', 'genre', 'loan_date', 'return_date',
       'late_fee', 'returned'],
      dtype='str')

In [55]:
import pandas as pd

loans_raw = pd.read_csv("data/loans.csv")  # original : Never touch it
loans = loans_raw.copy()  # working copy to clean

# remove space and lower
loans["genre"] = loans["genre"].str.strip().str.lower()

# replace same name repitition
loans["genre"] = loans["genre"].replace({
    "children's" : "children",
    "science fiction" : "sci-fi",
    "sci fi" : "sci-fi"
})

# use title()
loans["genre"] = loans["genre"].str.title()

# remove space and lower
loans["returned"] = loans["returned"].str.strip().str.lower()

# map yes or no to true false values
loans["returned"] = loans["returned"].map({"yes": True, "y": True, "no": False, "n": False})

# replace kr with empty
loans["late_fee"]= loans["late_fee"].str.replace("kr", "")

# remove space and replace , with .
loans["late_fee"] = loans["late_fee"].str.strip()
loans["late_fee"]= loans["late_fee"].str.replace(",", ".")

# convert str type to float type
loans["late_fee"] = loans["late_fee"].astype(float)

# pd.to_datetime() turns text into real dates
# errors="coerce"	if a value can’t be read as a date, put NaT instead of stopping with an error
# NaT	“Not a Time”. It’s NaN for dates: an empty date cell
loans["loan_date"] = pd.to_datetime(loans["loan_date"], errors='coerce')

# drop the duplicates
loans = loans.drop_duplicates()

# reset the index
loans = loans.reset_index(drop=True)

# remove column with no values
loans = loans.drop(columns="notes")

loans.to_csv("data/loans_clean.csv", index=False)

loans.shape

(605, 8)

### Answer – Task 7

**What I did:**
1. `drop(columns="notes")` removed the empty `notes` column.
2. I put every cleaning step in **one cell**, in the same order as Tasks 2–7:
   read the raw CSV → copy → genre → returned → late_fee → loan_date → drop duplicates + reset index → drop notes.
   Only lines that **change** the table are in it; checks like `unique()`, `describe()` and `sum()` stay in their task cells.
3. `to_csv("data/loans_clean.csv", index=False)` saved the clean table. `index=False` keeps the row numbers out of the file.
4. `shape` is the last line, so the cell shows it: **(605, 8)**.

**Does it work when I run it twice?** Yes, it gives (605, 8) both times.

**Why?** The cell starts with `pd.read_csv("data/loans.csv")` and `loans_raw.copy()`, so every run builds `loans` again from the untouched raw file. The `notes` column is always there when the drop runs.

A cell with only `loans = loans.drop(columns="notes")` fails the second time (KeyError), because it starts from the already changed `loans` and `notes` is already gone.